In [ ]:
from collections import Counter
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================================================================
# 0. MAPPER: ΦΟΡΤΩΣΗ ESCO OCCUPATIONS ΓΙΑ ΜΕΤΑΤΡΟΠΗ URIS ΣΕ ΟΝΟΜΑΤΑ
# ==============================================================================
esco_occ_map = {}
if os.path.exists("occupations_en.csv"):
    df_esco_occ = pd.read_csv("occupations_en.csv")
    # Δημιουργία λεξικού: conceptUri -> preferredLabel
    esco_occ_map = dict(
        zip(df_esco_occ["conceptUri"], df_esco_occ["preferredLabel"])
    )
    print(f"✅ Φορτώθηκαν {len(esco_occ_map)} αντιστοιχίσεις επαγγελμάτων ESCO.")


def get_occ_name(uri):
    """Επιστρέφει το όνομα του επαγγέλματος αν υπάρχει στο ESCO, αλλιώς το URI/ISCO"""
    if uri in esco_occ_map:
        return esco_occ_map[uri]
    # Αν είναι ISCO code (π.χ. C2433)
    if "isco" in uri:
        return f"ISCO Group: {uri.split('/')[-1]}"
    return uri.split("/")[-1]  # Επιστρέφει το ID


def get_skill_name(uri):
    """Καθαρίζει το Skill URI για καλύτερη εμφάνιση"""
    return uri.split("/")[-1][:8] + "..." if "/" in uri else uri


# ==============================================================================
# 1. ΦΟΡΤΩΣΗ ΚΑΙ ΤΩΝ 36 DATASETS (ME SAFETY CHECKS)
# ==============================================================================
BASE_DIR = "."  # /content στο Colab

COUNTRIES = [
    "czechia",
    "france",
    "germany",
    "greece",
    "italy",
    "poland",
    "romania",
    "spain",
    "sweden",
]
CATEGORIES = ["green", "lignite"]
KINDS = ["jobs", "profiles"]


def load_records_safe(path):
    """Loader με έλεγχο τύπων για αποφυγή AttributeError"""
    if not os.path.exists(path):
        # Δοκιμή με ή χωρίς .json επέκταση
        if os.path.exists(f"{path}.json"):
            path = f"{path}.json"
        else:
            return []

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        data_ = json.loads(content)
        if isinstance(data_, list):
            return [x for x in data_ if isinstance(x, dict)]
        elif isinstance(data_, dict):
            for v in data_.values():
                if isinstance(v, list):
                    return [x for x in v if isinstance(x, dict)]
            return [data_]
    except json.JSONDecodeError:
        pass

    records = []
    for line in content.splitlines():
        line = line.strip().rstrip(",")
        if not line or line in ("[", "]", "{", "}"):
            continue
        try:
            item = json.loads(line)
            if isinstance(item, dict):
                records.append(item)
        except json.JSONDecodeError:
            continue
    return records


# Φόρτωση
data = {"jobs": {"green": {}, "lignite": {}}, "profiles": {"green": {}, "lignite": {}}}

for kind in KINDS:
    for category in CATEGORIES:
        for country in COUNTRIES:
            p = f"{BASE_DIR}/{kind}_{country}_{category}_clean.json"
            data[kind][category][country] = load_records_safe(p)

print("✅ Η ασφαλής φόρτωση των 36 datasets ολοκληρώθηκε!\n")

# ==============================================================================
# 2. TOP-5 ZΗΤΟΥΜΕΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (JOBS) ΑΝΑ ΧΩΡΑ
# ==============================================================================
print("=" * 80)
print(" 1. TOP-5 ZΗΤΟΥΜΕΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (SKILLS) ΑΝΑ ΧΩΡΑ (JOBS) ")
print("=" * 80)

for country in COUNTRIES:
    print(f"\n📍 ΧΩΡΑ: {country.upper()}")
    for cat in CATEGORIES:
        jobs = data["jobs"][cat].get(country, [])
        skills_list = [
            s
            for j in jobs
            if isinstance(j, dict)
            for s in j.get("skills", [])
            if s
        ]
        top_skills = Counter(skills_list).most_common(5)

        print(f"  ► [{cat.upper()} JOBS] (Σύνολο Αναφορών: {len(skills_list)})")
        if not top_skills:
            print("     Δεν βρέθηκαν skills.")
        for sk_uri, count in top_skills:
            print(f"     • {get_skill_name(sk_uri)} ({sk_uri}): {count} αναφορές")

# ==============================================================================
# 3. TOP-5 ΔΕΞΙΟΤΗΤΕΣ ΕΡΓΑΖΟΜΕΝΩΝ (PROFILES) ΑΝΑ ΧΩΡΑ
# ==============================================================================
print("\n" + "=" * 80)
print(" 2. TOP-5 ΔΕΞΙΟΤΗΤΕΣ ΕΡΓΑΖΟΜΕΝΩΝ ΑΝΑ ΧΩΡΑ (PROFILES) ")
print("=" * 80)

for country in COUNTRIES:
    print(f"\n📍 ΧΩΡΑ: {country.upper()}")
    for cat in CATEGORIES:
        profiles = data["profiles"][cat].get(country, [])
        skills_list = [
            s
            for prof in profiles
            if isinstance(prof, dict)
            for s in prof.get("skills", [])
            if s
        ]
        top_skills = Counter(skills_list).most_common(5)

        print(
            f"  ► [{cat.upper()} PROFILES] (Σύνολο Αναφορών: {len(skills_list)})"
        )
        if not top_skills:
            print("     Δεν βρέθηκαν skills.")
        for sk_uri, count in top_skills:
            print(f"     • {get_skill_name(sk_uri)} ({sk_uri}): {count} αναφορές")

# ==============================================================================
# 4. TOP-5 OCCUPATIONS ΣΤΙΣ ΑΓΓΕΛΙΕΣ (JOBS) ME ΑΝΤΙΣΤΟΙΧΙΣΗ ΟΝΟΜΑΤΩΝ ESCO
# ==============================================================================
print("\n" + "=" * 80)
print(" 3. TOP-5 ΠΙΟ ΣΥΧΝΑ ΕΠΑΓΓΕΛΜΑΤΑ (ESCO OCCUPATIONS) ΣΤΙΣ ΑΓΓΕΛΙΕΣ (JOBS) ")
print("=" * 80)

for country in COUNTRIES:
    print(f"\n📍 ΧΩΡΑ: {country.upper()}")
    for cat in CATEGORIES:
        jobs = data["jobs"][cat].get(country, [])
        occs_list = [
            o
            for j in jobs
            if isinstance(j, dict)
            for o in j.get("occupations", [])
            if o
        ]
        top_occs = Counter(occs_list).most_common(5)

        print(f"  ► [{cat.upper()} JOBS]")
        if not top_occs:
            print("     Δεν βρέθηκαν occupations.")
        for occ_uri, count in top_occs:
            occ_name = get_occ_name(occ_uri)
            print(f"     • {occ_name}: {count} αγγελίες")

# ==============================================================================
# 5. ΣΥΓΚΡΙΤΙΚΗ ΑΝΑΛΥΣΗ: GREEN VS LIGNITE JOBS & COMMON SKILLS
# ==============================================================================
print("\n" + "=" * 80)
print(" 4. ΣΥΓΚΡΙΤΙΚΗ ΑΝΑΛΥΣΗ: GREEN VS LIGNITE JOBS & ΚΟΙΝΕΣ ΔΕΞΙΟΤΗΤΕΣ ")
print("=" * 80)

transition_stats = []

for country in COUNTRIES:
    n_green = len(data["jobs"]["green"].get(country, []))
    n_lignite = len(data["jobs"]["lignite"].get(country, []))
    ratio = round(n_green / n_lignite, 2) if n_lignite > 0 else np.nan

    green_skills = set(
        s
        for j in data["jobs"]["green"].get(country, [])
        if isinstance(j, dict)
        for s in j.get("skills", [])
        if s
    )
    lignite_skills = set(
        s
        for j in data["jobs"]["lignite"].get(country, [])
        if isinstance(j, dict)
        for s in j.get("skills", [])
        if s
    )
    common_skills = green_skills.intersection(lignite_skills)

    transition_stats.append(
        {
            "Country": country.capitalize(),
            "Green Jobs": n_green,
            "Lignite Jobs": n_lignite,
            "Green/Lignite Ratio": ratio,
            "Common Skills": len(common_skills),
        }
    )

df_trans = pd.DataFrame(transition_stats)
display(df_trans)

# ==============================================================================
# 6. SKILL MATCHING SCORE & GAP ANALYSIS (PROFILES VS JOBS)
# ==============================================================================
print("\n" + "=" * 80)
print(" 5. SKILL MATCHING SCORE & GAP ANALYSIS (LIGNITE PROFILES VS GREEN JOBS) ")
print("=" * 80)

match_results = []

for country in COUNTRIES:
    lignite_prof_skills = set(
        s
        for p in data["profiles"]["lignite"].get(country, [])
        if isinstance(p, dict)
        for s in p.get("skills", [])
        if s
    )

    green_job_skills_counter = Counter(
        [
            s
            for j in data["jobs"]["green"].get(country, [])
            if isinstance(j, dict)
            for s in j.get("skills", [])
            if s
        ]
    )
    green_job_skills_set = set(green_job_skills_counter.keys())

    if len(lignite_prof_skills) > 0:
        matched = lignite_prof_skills.intersection(green_job_skills_set)
        score = round((len(matched) / len(lignite_prof_skills)) * 100, 2)
    else:
        score = 0.0

    match_results.append(
        {
            "Country": country.capitalize(),
            "Lignite Profile Skills": len(lignite_prof_skills),
            "Green Job Skills Needed": len(green_job_skills_set),
            "Skill Matching Score (%)": score,
        }
    )

df_match = pd.DataFrame(match_results)
display(df_match)

✅ Η ασφαλής φόρτωση των 36 datasets ολοκληρώθηκε!

 1. TOP-5 ZΗΤΟΥΜΕΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (SKILLS) ΑΝΑ ΧΩΡΑ (JOBS) 

📍 ΧΩΡΑ: CZECHIA
  ► [GREEN JOBS] (Σύνολο Αναφορών: 1291)
     • 199f7919... (http://data.europa.eu/esco/skill/199f7919-5114-41b6-b6a5-41e0e4896ec1): 47 αναφορές
     • 3de6d427... (http://data.europa.eu/esco/skill/3de6d427-546b-450b-97ba-d3d50eb274da): 36 αναφορές
     • 5b06c369... (http://data.europa.eu/esco/skill/5b06c369-10db-477d-a8df-987179324001): 28 αναφορές
     • f806d624... (http://data.europa.eu/esco/skill/f806d624-7700-4e47-9f7b-253d4d6ab9e5): 24 αναφορές
     • db9050db... (http://data.europa.eu/esco/skill/db9050db-ae5d-4be8-8114-c26f272c3bda): 21 αναφορές
  ► [LIGNITE JOBS] (Σύνολο Αναφορών: 488)
     • c4fe78ae... (http://data.europa.eu/esco/skill/c4fe78ae-8fef-4aa6-b767-146ed2b40198): 82 αναφορές
     • 35f1dcdc... (http://data.europa.eu/esco/skill/35f1dcdc-577f-46c6-96d9-6c7a64501de9): 20 αναφορές
     • 199f7919... (http://data.europa.eu/esco/skill/199f7919-5

,Country,Green Jobs,Lignite Jobs,Green/Lignite Ratio,Common Skills
0,Czechia,371,237,1.57,80
1,France,40702,38154,1.07,3572
2,Germany,23789,48009,0.50,3469
3,Greece,3624,2887,1.26,1260
4,Italy,246,514,0.48,85
5,Poland,545,391,1.39,134
6,Romania,76,0,NaN,0
7,Spain,898,965,0.93,149
8,Sweden,34397,24004,1.43,3702



 5. SKILL MATCHING SCORE & GAP ANALYSIS (LIGNITE PROFILES VS GREEN JOBS) 


,Country,Lignite Profile Skills,Green Job Skills Needed,Skill Matching Score (%)
0,Czechia,531,569,23.16
1,France,1567,5175,88.39
2,Germany,1767,4782,72.95
3,Greece,894,2320,68.90
4,Italy,1669,309,11.20
5,Poland,1072,517,18.10
6,Romania,837,61,2.99
7,Spain,1583,586,16.93
8,Sweden,1050,5402,94.29


#claude

In [4]:
from collections import Counter
import json
import os
import numpy as np
import pandas as pd

# ==============================================================================
# 0. ΡΥΘΜΙΣΕΙΣ
# ==============================================================================
# Colab: τα αρχεία που ανεβάζεις εμφανίζονται συνήθως στο /content
BASE_DIR = "/content"

COUNTRIES = ["czechia", "france", "germany", "greece", "italy",
             "poland", "romania", "spain", "sweden"]
CATEGORIES = ["green", "lignite"]
KINDS = ["jobs", "profiles"]

# ==============================================================================
# 1. MAPPERS: ΦΟΡΤΩΣΗ CSV ΓΙΑ ΜΕΤΑΤΡΟΠΗ URIs ΣΕ ΑΝΑΓΝΩΣΙΜΑ ΟΝΟΜΑΤΑ
# ==============================================================================
def build_label_map(path):
    """Διαβάζει ένα ESCO-style CSV και φτιάχνει dict URI -> preferredLabel.
    Εντοπίζει αυτόματα τις σωστές στήλες ακόμη κι αν δεν λέγονται ακριβώς
    'conceptUri' / 'preferredLabel'."""
    if not os.path.exists(path):
        print(f"⚠️ Δεν βρέθηκε: {path} — τα URIs αυτής της κατηγορίας θα εμφανίζονται ως έχουν.")
        return {}
    df_ref = pd.read_csv(path)

    uri_col = None
    for col in df_ref.columns:
        sample = df_ref[col].dropna().astype(str).head(20)
        if sample.str.startswith("http://data.europa.eu/esco/").any():
            uri_col = col
            break

    label_col = None
    for col in df_ref.columns:
        if "preferredlabel" in col.lower().replace("_", ""):
            label_col = col
            break
    if label_col is None:
        # fallback: πρώτη στήλη τύπου string που δεν είναι η uri_col
        for col in df_ref.columns:
            if col != uri_col and df_ref[col].dtype == object:
                label_col = col
                break

    if uri_col is None or label_col is None:
        print(f"⚠️ Δεν εντοπίστηκαν στήλες URI/label στο {path}. Στήλες: {list(df_ref.columns)}")
        return {}

    mapping = dict(zip(df_ref[uri_col].astype(str).str.strip(), df_ref[label_col]))
    print(f"✅ {path}: φορτώθηκαν {len(mapping)} αντιστοιχίσεις (στήλες: '{uri_col}' -> '{label_col}').")
    return mapping

# --- Occupations: συνδυάζουμε τα δύο δικά σου curated αρχεία ---
green_occ_map = build_label_map(f"{BASE_DIR}/highly_green_occupations.csv")
lignite_occ_map = build_label_map(f"{BASE_DIR}/lignite_occupations.csv")
esco_occ_map = {**green_occ_map, **lignite_occ_map}
print(f"➡️ Σύνολο μοναδικών occupation labels διαθέσιμων: {len(esco_occ_map)}\n")

# --- Skills: προαιρετικό, αν έχεις κάποιο σχετικό αρχείο τοπικά θα αντιστοιχιστεί,
#     αλλιώς θα εμφανίζεται το κομμένο URI (όπως έκανε και ο αρχικός κώδικας) ---
esco_skill_map = {}
for candidate in ["skills_en.csv", "greenSkillsCollection_en.csv"]:
    m = build_label_map(f"{BASE_DIR}/{candidate}")
    esco_skill_map.update(m)
print(f"➡️ Σύνολο μοναδικών skill labels διαθέσιμων: {len(esco_skill_map)}\n")

def is_isco_uri(uri):
    return "/esco/isco/" in uri

def get_occ_name(uri):
    if uri in esco_occ_map:
        return esco_occ_map[uri]
    if is_isco_uri(uri):
        return f"ISCO Code: {uri.split('/')[-1]}"
    return uri.split("/")[-1]

def get_skill_name(uri):
    if uri in esco_skill_map:
        return esco_skill_map[uri]
    return uri.split("/")[-1][:8] + "..." if "/" in uri else uri

# ==============================================================================
# 2. ΦΟΡΤΩΣΗ ΚΑΙ ΤΩΝ 36 DATASETS (ΜΕ SAFETY CHECKS)
# ==============================================================================
def load_records_safe(path):
    if not os.path.exists(path):
        if os.path.exists(f"{path}.json"):
            path = f"{path}.json"
        else:
            return []
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    try:
        data_ = json.loads(content)
        if isinstance(data_, list):
            return [x for x in data_ if isinstance(x, dict)]
        elif isinstance(data_, dict):
            for v in data_.values():
                if isinstance(v, list):
                    return [x for x in v if isinstance(x, dict)]
            return [data_]
    except json.JSONDecodeError:
        pass
    records = []
    for line in content.splitlines():
        line = line.strip().rstrip(",")
        if not line or line in ("[", "]", "{", "}"):
            continue
        try:
            item = json.loads(line)
            if isinstance(item, dict):
                records.append(item)
        except json.JSONDecodeError:
            continue
    return records

data = {"jobs": {"green": {}, "lignite": {}}, "profiles": {"green": {}, "lignite": {}}}
for kind in KINDS:
    for category in CATEGORIES:
        for country in COUNTRIES:
            p = f"{BASE_DIR}/{kind}_{country}_{category}_clean.json"
            data[kind][category][country] = load_records_safe(p)

print("✅ Η φόρτωση των 36 datasets ολοκληρώθηκε!\n")

# Γρήγορο overview μεγεθών datasets
summary_rows = [
    {"kind": kind, "category": category, "country": country, "n_records": len(recs)}
    for kind in data for category in data[kind] for country, recs in data[kind][category].items()
]
df_summary = pd.DataFrame(summary_rows)
display(df_summary.pivot_table(index=["kind", "category"], columns="country", values="n_records"))

✅ /content/highly_green_occupations.csv: φορτώθηκαν 152 αντιστοιχίσεις (στήλες: 'conceptUri' -> 'preferredLabel').
✅ /content/lignite_occupations.csv: φορτώθηκαν 122 αντιστοιχίσεις (στήλες: 'conceptUri' -> 'preferredLabel').
➡️ Σύνολο μοναδικών occupation labels διαθέσιμων: 263

⚠️ Δεν βρέθηκε: /content/skills_en.csv — τα URIs αυτής της κατηγορίας θα εμφανίζονται ως έχουν.
✅ /content/greenSkillsCollection_en.csv: φορτώθηκαν 629 αντιστοιχίσεις (στήλες: 'conceptUri' -> 'preferredLabel').
➡️ Σύνολο μοναδικών skill labels διαθέσιμων: 629

✅ Η φόρτωση των 36 datasets ολοκληρώθηκε!



country            czechia   france  germany  greece    italy  poland  \
kind     category                                                       
jobs     green       371.0  40702.0  23789.0  3624.0    246.0   545.0   
         lignite     237.0  38154.0  48009.0  2887.0    514.0   391.0   
profiles green      3064.0  54369.0  34557.0  4913.0  37320.0  8744.0   
         lignite    3036.0  45233.0  25707.0  3247.0  23290.0  8731.0   

country            romania    spain   sweden  
kind     category                             
jobs     green        76.0    898.0      0.0  
         lignite       0.0    965.0  24004.0  
profiles green      7997.0  27707.0  10136.0  
         lignite    5157.0  21843.0   9100.0

In [5]:
# ==============================================================================
# 1. TOP-5 ZΗΤΟΥΜΕΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (JOBS) ΑΝΑ ΧΩΡΑ — green/lignite ξεχωριστά
# ==============================================================================
print("=" * 80)
print(" 1. TOP-5 ΖΗΤΟΥΜΕΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (SKILLS) ΑΝΑ ΧΩΡΑ (JOBS) ")
print("=" * 80)

for country in COUNTRIES:
    print(f"\n📍 ΧΩΡΑ: {country.upper()}")
    for cat in CATEGORIES:
        jobs = data["jobs"][cat].get(country, [])
        skills_list = [s for j in jobs for s in j.get("skills", []) if s]
        top_skills = Counter(skills_list).most_common(5)
        print(f"  ► [{cat.upper()} JOBS] (Σύνολο αναφορών: {len(skills_list)})")
        if not top_skills:
            print("     Δεν βρέθηκαν skills.")
        for sk_uri, count in top_skills:
            print(f"     • {get_skill_name(sk_uri)}: {count} αναφορές")

# ==============================================================================
# 2. TOP-5 ΔΕΞΙΟΤΗΤΕΣ ΕΡΓΑΖΟΜΕΝΩΝ (PROFILES) ΑΝΑ ΧΩΡΑ + fill-rate έλεγχος
# ==============================================================================
print("\n" + "=" * 80)
print(" 2. TOP-5 ΔΕΞΙΟΤΗΤΕΣ ΕΡΓΑΖΟΜΕΝΩΝ ΑΝΑ ΧΩΡΑ (PROFILES) ")
print("=" * 80)

print("\n--- Fill-rate πεδίου 'skills' στα profiles (πόσο αξιόπιστο είναι το top-5) ---")
fill_rows = []
for cat in CATEGORIES:
    for country in COUNTRIES:
        profs = data["profiles"][cat].get(country, [])
        total = len(profs)
        with_skills = sum(1 for p in profs if p.get("skills"))
        pct = round(with_skills / total * 100, 1) if total else 0.0
        fill_rows.append({"category": cat, "country": country, "total": total,
                           "with_skills": with_skills, "fill_rate_%": pct})
df_fill = pd.DataFrame(fill_rows)
display(df_fill)

for country in COUNTRIES:
    print(f"\n📍 ΧΩΡΑ: {country.upper()}")
    for cat in CATEGORIES:
        profiles = data["profiles"][cat].get(country, [])
        skills_list = [s for prof in profiles for s in prof.get("skills", []) if s]
        top_skills = Counter(skills_list).most_common(5)
        print(f"  ► [{cat.upper()} PROFILES] (Σύνολο αναφορών: {len(skills_list)})")
        if not top_skills:
            print("     Δεν βρέθηκαν skills.")
        for sk_uri, count in top_skills:
            print(f"     • {get_skill_name(sk_uri)}: {count} αναφορές")

# ==============================================================================
# 3. TOP-5 ΣΥΓΚΕΚΡΙΜΕΝΑ OCCUPATIONS ΣΤΙΣ ΑΓΓΕΛΙΕΣ (εξαιρούνται τα ISCO group URIs)
# ==============================================================================
print("\n" + "=" * 80)
print(" 3. TOP-5 ΠΙΟ ΣΥΧΝΑ ΕΠΑΓΓΕΛΜΑΤΑ ΣΤΙΣ ΑΓΓΕΛΙΕΣ (JOBS) ")
print("=" * 80)

for country in COUNTRIES:
    print(f"\n📍 ΧΩΡΑ: {country.upper()}")
    for cat in CATEGORIES:
        jobs = data["jobs"][cat].get(country, [])
        occs_list = [o for j in jobs for o in j.get("occupations", [])
                     if o and not is_isco_uri(o)]
        top_occs = Counter(occs_list).most_common(5)
        print(f"  ► [{cat.upper()} JOBS]")
        if not top_occs:
            print("     Δεν βρέθηκαν occupations.")
        for occ_uri, count in top_occs:
            print(f"     • {get_occ_name(occ_uri)}: {count} αγγελίες")

# ==============================================================================
# 4. ΚΛΑΔΟΙ (SECTORS): ΠΟΙΟΙ ΚΥΡΙΑΡΧΟΥΝ ΣΤΟ GREEN vs LIGNITE ΑΝΑ ΧΩΡΑ
# ==============================================================================
print("\n" + "=" * 80)
print(" 4. ΣΥΓΚΡΙΣΗ ΚΛΑΔΩΝ (SECTORS): GREEN VS LIGNITE ")
print("=" * 80)

for country in COUNTRIES:
    green_sectors = Counter(s for j in data["jobs"]["green"].get(country, [])
                             for s in j.get("sectors", []) if s)
    lignite_sectors = Counter(s for j in data["jobs"]["lignite"].get(country, [])
                               for s in j.get("sectors", []) if s)
    all_sectors = set(green_sectors) | set(lignite_sectors)
    ranked = sorted(all_sectors,
                     key=lambda s: green_sectors.get(s, 0) - lignite_sectors.get(s, 0),
                     reverse=True)

    print(f"\n📍 {country.upper()}")
    print("  Κλάδοι που κυριαρχούν στο GREEN:")
    for s in ranked[:5]:
        print(f"     • {s}: green={green_sectors.get(s,0)} vs lignite={lignite_sectors.get(s,0)}")
    print("  Κλάδοι που κυριαρχούν στο LIGNITE:")
    for s in reversed(ranked[-5:]):
        print(f"     • {s}: lignite={lignite_sectors.get(s,0)} vs green={green_sectors.get(s,0)}")

# ==============================================================================
# 5. ΣΥΓΚΡΙΤΙΚΗ ΑΝΑΛΥΣΗ: GREEN VS LIGNITE JOBS (λόγος + κοινές δεξιότητες)
# ==============================================================================
print("\n" + "=" * 80)
print(" 5. GREEN VS LIGNITE JOBS — ΛΟΓΟΣ & ΚΟΙΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (ΓΕΦΥΡΕΣ) ")
print("=" * 80)

transition_stats = []
for country in COUNTRIES:
    n_green = len(data["jobs"]["green"].get(country, []))
    n_lignite = len(data["jobs"]["lignite"].get(country, []))
    ratio = round(n_green / n_lignite, 2) if n_lignite > 0 else np.nan

    green_skills = set(s for j in data["jobs"]["green"].get(country, [])
                        for s in j.get("skills", []) if s)
    lignite_skills = set(s for j in data["jobs"]["lignite"].get(country, [])
                          for s in j.get("skills", []) if s)
    common_skills = green_skills & lignite_skills

    transition_stats.append({
        "Country": country.capitalize(),
        "Green Jobs": n_green,
        "Lignite Jobs": n_lignite,
        "Green/Lignite Ratio": ratio,
        "Common Skills (count)": len(common_skills),
    })

df_trans = pd.DataFrame(transition_stats)
display(df_trans)

print("\n--- Ονόματα δεξιοτήτων-γεφυρών (κοινές lignite ↔ green jobs) ---")
for country in COUNTRIES:
    green_skills = set(s for j in data["jobs"]["green"].get(country, [])
                        for s in j.get("skills", []) if s)
    lignite_skills = set(s for j in data["jobs"]["lignite"].get(country, [])
                          for s in j.get("skills", []) if s)
    common = green_skills & lignite_skills
    print(f"\n📍 {country.upper()} — {len(common)} κοινές δεξιότητες (top 10):")
    for uri in list(common)[:10]:
        print(f"   • {get_skill_name(uri)}")

# ==============================================================================
# 6. SKILL MATCHING SCORE & GAP ANALYSIS (LIGNITE PROFILES VS GREEN JOBS)
# ==============================================================================
print("\n" + "=" * 80)
print(" 6. SKILL MATCHING SCORE & GAP ANALYSIS (LIGNITE PROFILES VS GREEN JOBS) ")
print("=" * 80)

match_results = []
gap_rows = []

for country in COUNTRIES:
    lignite_prof_skills = set(s for p in data["profiles"]["lignite"].get(country, [])
                               for s in p.get("skills", []) if s)

    green_job_skills_counter = Counter(
        s for j in data["jobs"]["green"].get(country, [])
        for s in j.get("skills", []) if s
    )
    green_job_skills_set = set(green_job_skills_counter.keys())

    if len(lignite_prof_skills) > 0:
        matched = lignite_prof_skills & green_job_skills_set
        score = round((len(matched) / len(lignite_prof_skills)) * 100, 2)
    else:
        score = np.nan  # δεν υπάρχουν δεδομένα -> άγνωστο, όχι 0%

    match_results.append({
        "Country": country.capitalize(),
        "Lignite Profile Skills": len(lignite_prof_skills),
        "Green Job Skills Needed": len(green_job_skills_set),
        "Skill Matching Score (%)": score,
    })

    # Gap analysis: skills που ζητούνται πολύ σε green jobs αλλά λείπουν από lignite profiles
    gaps = [(uri, cnt) for uri, cnt in green_job_skills_counter.most_common()
            if uri not in lignite_prof_skills]
    for uri, cnt in gaps[:10]:
        gap_rows.append({
            "Country": country.capitalize(),
            "Missing Skill": get_skill_name(uri),
            "Demand in Green Jobs": cnt,
        })

df_match = pd.DataFrame(match_results)
print("\n--- Skill Matching Score (ετοιμότητα lignite → green) ---")
display(df_match.sort_values("Skill Matching Score (%)", ascending=False))

df_gap = pd.DataFrame(gap_rows)
print("\n--- Top Skill Gaps ανά χώρα (μεγάλη ζήτηση, απούσα από lignite profiles) ---")
for country in COUNTRIES:
    sub = df_gap[df_gap["Country"] == country.capitalize()]
    if sub.empty:
        continue
    print(f"\n📍 {country.upper()}")
    display(sub.reset_index(drop=True))

 1. TOP-5 ΖΗΤΟΥΜΕΝΕΣ ΔΕΞΙΟΤΗΤΕΣ (SKILLS) ΑΝΑ ΧΩΡΑ (JOBS) 

📍 ΧΩΡΑ: CZECHIA
  ► [GREEN JOBS] (Σύνολο αναφορών: 1291)
     • 199f7919...: 47 αναφορές
     • operate water purifying equipment: 36 αναφορές
     • manage water distribution procedures: 28 αναφορές
     • promote animal welfare: 24 αναφορές
     • manage health and safety: 21 αναφορές
  ► [LIGNITE JOBS] (Σύνολο αναφορών: 488)
     • c4fe78ae...: 82 αναφορές
     • 35f1dcdc...: 20 αναφορές
     • 199f7919...: 19 αναφορές
     • c5bc42be...: 15 αναφορές
     • 8d1dc94c...: 13 αναφορές

📍 ΧΩΡΑ: FRANCE
  ► [GREEN JOBS] (Σύνολο αναφορών: 221742)
     • a06a0d41...: 4953 αναφορές
     • c5bc42be...: 3139 αναφορές
     • 2b1ce548...: 2876 αναφορές
     • f3c5fae0...: 2786 αναφορές
     • 91abe492...: 2471 αναφορές
  ► [LIGNITE JOBS] (Σύνολο αναφορών: 165605)
     • c5bc42be...: 4995 αναφορές
     • a6354281...: 3462 αναφορές
     • 2b1ce548...: 2740 αναφορές
     • bf419e00...: 2453 αναφορές
     • f3c5fae0...: 1883 αναφορές

📍 ΧΩΡΑ

,category,country,total,with_skills,fill_rate_%
0,green,czechia,3064,328,10.7
1,green,france,54369,3433,6.3
2,green,germany,34557,3216,9.3
3,green,greece,4913,1073,21.8
4,green,italy,37320,3428,9.2
5,green,poland,8744,1119,12.8
6,green,romania,7997,937,11.7
7,green,spain,27707,3215,11.6
8,green,sweden,10136,1174,11.6
9,lignite,czechia,3036,352,11.6



📍 ΧΩΡΑ: CZECHIA
  ► [GREEN PROFILES] (Σύνολο αναφορών: 1007)
     • cd5efa8c...: 21 αναφορές
     • 15d76317...: 13 αναφορές
     • bda0d115...: 12 αναφορές
     • urban sustainability: 11 αναφορές
     • 9531fe02...: 10 αναφορές
  ► [LIGNITE PROFILES] (Σύνολο αναφορών: 999)
     • cd5efa8c...: 18 αναφορές
     • ccd0a1d9...: 14 αναφορές
     • f4a6e9f7...: 12 αναφορές
     • f0de4973...: 11 αναφορές
     • 75b30aeb...: 10 αναφορές

📍 ΧΩΡΑ: FRANCE
  ► [GREEN PROFILES] (Σύνολο αναφορών: 8108)
     • cd5efa8c...: 170 αναφορές
     • 15d76317...: 101 αναφορές
     • 2fb8480e...: 97 αναφορές
     • 19a8293b...: 81 αναφορές
     • ccd0a1d9...: 76 αναφορές
  ► [LIGNITE PROFILES] (Σύνολο αναφορών: 6106)
     • cd5efa8c...: 108 αναφορές
     • 2fb8480e...: 93 αναφορές
     • ccd0a1d9...: 81 αναφορές
     • 15d76317...: 69 αναφορές
     • c5bc42be...: 59 αναφορές

📍 ΧΩΡΑ: GERMANY
  ► [GREEN PROFILES] (Σύνολο αναφορών: 9257)
     • cd5efa8c...: 212 αναφορές
     • urban sustainability: 135 αναφ

,Country,Green Jobs,Lignite Jobs,Green/Lignite Ratio,Common Skills (count)
0,Czechia,371,237,1.57,80
1,France,40702,38154,1.07,3572
2,Germany,23789,48009,0.50,3469
3,Greece,3624,2887,1.26,1260
4,Italy,246,514,0.48,85
5,Poland,545,391,1.39,134
6,Romania,76,0,NaN,0
7,Spain,898,965,0.93,149
8,Sweden,0,24004,0.00,0



--- Ονόματα δεξιοτήτων-γεφυρών (κοινές lignite ↔ green jobs) ---

📍 CZECHIA — 80 κοινές δεξιότητες (top 10):
   • 001115fb...
   • 15d76317...
   • 0b3ee7c8...
   • 2d9aaad3...
   • 0f0698e7...
   • 00e895f4...
   • 50f5e41d...
   • 93a68dcb...
   • 5f95f9c9...
   • 7c632656...

📍 FRANCE — 3572 κοινές δεξιότητες (top 10):
   • 6b222aac...
   • 614fdb7f...
   • e1479957...
   • 2a0efd9a...
   • 3d96dc4c...
   • 26792722...
   • c7d8f44f...
   • 7ec3c77b...
   • f414dc08...
   • ec7ec825...

📍 GERMANY — 3469 κοινές δεξιότητες (top 10):
   • 50dd3681...
   • 614fdb7f...
   • e1479957...
   • 2a0efd9a...
   • ab580ada...
   • 62d34066...
   • daefc625...
   • 26514334...
   • 65df6a1d...
   • 1612575c...

📍 GREECE — 1260 κοινές δεξιότητες (top 10):
   • 614fdb7f...
   • 2a0efd9a...
   • db36f7f1...
   • a6d697d1...
   • f414dc08...
   • ec7ec825...
   • b4877620...
   • dfb9cec6...
   • f14ff4b7...
   • e5beeff8...

📍 ITALY — 85 κοινές δεξιότητες (top 10):
   • 6c153605...
   • b011c8b4..

...

,Country,Missing Skill,Demand in Green Jobs
0,Poland,design ventilation network,21
1,Poland,environmental engineering,16
2,Poland,204713d1...,16
3,Poland,develop waste management processes,14
4,Poland,spatial planning,13
5,Poland,develop sewerage networks,10
6,Poland,maintain waste collection records,10
7,Poland,811acf0c...,10
8,Poland,manage water distribution procedures,10
9,Poland,6d97bf55...,10



📍 ROMANIA


,Country,Missing Skill,Demand in Green Jobs
0,Romania,social entreprise,18
1,Romania,disinfect surfaces,3
2,Romania,design ventilation network,2
3,Romania,c662be22...,2
4,Romania,collect domestic waste,1
5,Romania,sustainable manufacturing,1
6,Romania,8f6ed69b...,1
7,Romania,f206e732...,1
8,Romania,manage waste,1
9,Romania,inspect industrial equipment,1



📍 SPAIN


,Country,Missing Skill,Demand in Green Jobs
0,Spain,disinfect surfaces,19
1,Spain,e1479957...,17
2,Spain,develop irrigation strategies,16
3,Spain,b7cf2d84...,13
4,Spain,5f95f9c9...,13
5,Spain,cb073ba6...,11
6,Spain,18a5e674...,10
7,Spain,maintain waste collection records,7
8,Spain,5125b132...,7
9,Spain,aa43f245...,7
